# Лабораторная 2: Transformer encoder на `IMDB`


## Что нужно знать до старта

Перед началом этой ЛР полезно открыть:
- [../README.md](../README.md)
- [guides/00_transformer_prerequisites.md](../guides/00_transformer_prerequisites.md)
- [guides/01_self_attention_and_positional_encoding_beginner.md](../guides/01_self_attention_and_positional_encoding_beginner.md)
- [guides/03_transformer_encoder_imdb_walkthrough.md](../guides/03_transformer_encoder_imdb_walkthrough.md)
- [../../00-Foundations/showcases/cards/01_imdb.md](../../../00-Foundations/showcases/cards/01_imdb.md)

Это вторая лабораторная блока `03-Transformer` и Шаг 6 общего курса.


## Интуиция задачи без формул

Во второй лабораторной меняется не идея архитектуры, а тип данных.

Теперь на входе:
- реальные review-последовательности из `IMDB`;
- после padding они всё ещё имеют форму `(N, T)`.

На выходе:
- одна бинарная sentiment-метка на всю последовательность.

То есть мы берём тот же encoder block и переносим его из synthetic мира в реальный text classification.


## Как проходить эту ЛР без преподавателя

Фиксированный порядок для темы `03-Transformer`:
1. Убедиться, что `03-Transformer / ЛР01` уже пройдена.
2. Прочитать `guides/03_transformer_encoder_imdb_walkthrough.md`.
3. Сделать свою первую попытку в этом starter notebook.
4. Если застряли, открыть `guides/04_transformer_debugging_playbook.md`.
5. Сделать вторую попытку после debugging-step.
6. Только потом смотреть в `solutions/02_transformer_encoder_imdb_solution.ipynb`.


## Что изменилось после `03-Transformer / ЛР01`

В прошлой работе идея была изолированной:
- короткие synthetic sequence;
- метка зависела от порядка специальных токенов.

Теперь:
- данные реальные и шумные;
- токенов намного больше;
- последовательности длиннее;
- attention trace нужно читать осторожнее и только на осмысленном фрагменте review.


## Контракт данных

Вход:
- padded review-последовательности token ids,
- форма `X -> (N, T)`.

Выход:
- бинарная sentiment-метка `y -> (N,)`.

Здесь `PAD = 0`, поэтому padding mask можно строить как `tokens != 0`.


## Таблица форм тензоров

| Сущность | Смысл | Форма |
|---|---|---|
| `X_train` | padded review tokens | `(N, T)` |
| `padding_mask` | полезные позиции | `(N, T)` |
| `embeddings` | token + position embeddings | `(N, T, E)` |
| `attention_scores` | веса внимания | `(N, H, T, T)` |
| `pooled` | один вектор на review | `(N, E)` |
| `y_pred` | вероятность positive class | `(N, 1)` |


## Шпаргалка по обозначениям и формам

- `N` — число review.
- `T` — длина review после padding.
- `E` — размер embedding / model dimension.
- `H` — число attention heads.
- `V` — размер словаря.

Для этой лабораторной важны две привычки:
- проверять mask;
- не пытаться интерпретировать attention на всём review целиком, если оно слишком длинное.


## Контракт модели

В этой работе reuse-block из `ЛР01` считается уже понятным и даётся готовым прямо в notebook:
1. `TokenAndPositionEmbedding`
2. `TransformerEncoderBlock`
3. `masked_average`

Новые задачи студента начинаются не с переписывания этих слоёв, а с такого набора шагов:
1. подготовить `IMDB` data;
2. собрать review classifier поверх reuse-block;
3. обучить и оценить модель;
4. снять один attention trace на review.


## Мини-теория

Encoder-only Transformer для классификации можно читать так:
- токены получают embeddings и позиции;
- encoder block смешивает информацию между позициями;
- затем всё сворачивается в один вектор на объект;
- classifier head выдаёт sentiment-вероятность.

На практике это уже рабочий пример того, как self-attention заменяет рекуррентный проход в `many-to-one` задаче.


## Ручной разбор одного примера

Review после tokenization — это уже не слова, а целочисленные ids.

С точки зрения формы всё выглядит знакомо:

```text
review tokens -> (T,)
mask          -> (T,)
embeddings    -> (T, E)
```

Новое здесь не shape, а то, что attention должен работать на реальном тексте, а не на короткой synthetic последовательности.


In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt


2026-03-24 11:59:41.388973: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-24 11:59:41.389317: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-24 11:59:41.391237: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-24 11:59:41.396666: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774342781.406891  280064 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774342781.40

In [2]:
SEED = 7
PAD_ID = 0
NUM_WORDS = 10000
MAXLEN = 200
TRAIN_SUBSET = 5000
TEST_SUBSET = 2000
EMBED_DIM = 32
NUM_HEADS = 2
FF_DIM = 64
BATCH_SIZE = 64
EPOCHS = 3

keras.utils.set_random_seed(SEED)
np.set_printoptions(linewidth=120)


In [3]:
(x_train_raw, y_train_raw), (x_test_raw, y_test_raw) = keras.datasets.imdb.load_data(num_words=NUM_WORDS)

x_train_raw = x_train_raw[:TRAIN_SUBSET]
y_train_raw = y_train_raw[:TRAIN_SUBSET]
x_test_raw = x_test_raw[:TEST_SUBSET]
y_test_raw = y_test_raw[:TEST_SUBSET]

X_train = keras.utils.pad_sequences(x_train_raw, maxlen=MAXLEN, padding='post', truncating='post')
X_test = keras.utils.pad_sequences(x_test_raw, maxlen=MAXLEN, padding='post', truncating='post')
y_train = np.asarray(y_train_raw, dtype=np.int32)
y_test = np.asarray(y_test_raw, dtype=np.int32)

print('X_train shape:', X_train.shape)
print('X_test shape :', X_test.shape)
print('positive rate:', y_train.mean())
print('non-pad length example:', int(np.count_nonzero(X_train[0])))


X_train shape: (5000, 200)
X_test shape : (2000, 200)
positive rate: 0.5092
non-pad length example: 200


In [4]:
print('class balance train:', np.bincount(y_train))
print('first padded tokens:', X_train[0][:25])
print('first useful length:', int(np.count_nonzero(X_train[0])))


class balance train: [2454 2546]
first padded tokens: [   1   14   22   16   43  530  973 1622 1385   65  458 4468   66 3941    4  173   36  256    5   25  100   43  838
  112   50]
first useful length: 200


In [5]:
class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.maxlen = maxlen
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.token_emb = layers.Embedding(vocab_size, embed_dim, mask_zero=True)
        self.pos_emb = layers.Embedding(maxlen, embed_dim)
        self.supports_masking = True

    def call(self, inputs):
        positions = tf.range(start=0, limit=tf.shape(inputs)[-1], delta=1)
        position_embeddings = self.pos_emb(positions)
        token_embeddings = self.token_emb(inputs)
        return token_embeddings + position_embeddings

    def compute_mask(self, inputs, mask=None):
        return self.token_emb.compute_mask(inputs)


def masked_average(x, mask):
    mask = tf.cast(mask, x.dtype)
    mask = tf.expand_dims(mask, axis=-1)
    summed = tf.reduce_sum(x * mask, axis=1)
    counts = tf.reduce_sum(mask, axis=1)
    return summed / tf.maximum(counts, 1.0)


class TransformerEncoderBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.ff_dim = ff_dim
        self.rate = rate
        self.att = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
            dropout=rate,
        )
        self.ffn = keras.Sequential(
            [
                layers.Dense(ff_dim, activation='relu'),
                layers.Dense(embed_dim),
            ]
        )
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)
        self.supports_masking = True

    def call(self, inputs, mask=None, training=None, return_attention_scores=False):
        attention_mask = None
        if mask is not None:
            attention_mask = tf.cast(mask[:, tf.newaxis, :], dtype=tf.bool)

        if return_attention_scores:
            attention_output, attention_scores = self.att(
                query=inputs,
                value=inputs,
                key=inputs,
                attention_mask=attention_mask,
                return_attention_scores=True,
                training=training,
            )
        else:
            attention_output = self.att(
                query=inputs,
                value=inputs,
                key=inputs,
                attention_mask=attention_mask,
                training=training,
            )

        attention_output = self.dropout1(attention_output, training=training)
        proj_input = self.layernorm1(inputs + attention_output)
        proj_output = self.ffn(proj_input)
        proj_output = self.dropout2(proj_output, training=training)
        output = self.layernorm2(proj_input + proj_output)

        if return_attention_scores:
            return output, attention_scores
        return output

    def compute_mask(self, inputs, mask=None):
        return mask


In [6]:
sample_embedding_layer = TokenAndPositionEmbedding(MAXLEN, NUM_WORDS, EMBED_DIM)
sample_encoder_block = TransformerEncoderBlock(EMBED_DIM, NUM_HEADS, FF_DIM)
sample_tokens = X_train[:2]
sample_mask = sample_tokens != PAD_ID
sample_embeddings = sample_embedding_layer(sample_tokens)
sample_encoded, sample_scores = sample_encoder_block(sample_embeddings, mask=sample_mask, return_attention_scores=True)

print('sample_embeddings:', sample_embeddings.shape)
print('sample_encoded   :', sample_encoded.shape)
print('sample_scores    :', sample_scores.shape)


sample_embeddings: (2, 200, 32)
sample_encoded   : (2, 200, 32)
sample_scores    : (2, 2, 200, 200)


E0000 00:00:1774342787.526210  280064 cuda_executor.cc:1228] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1774342787.528333  280064 gpu_device.cc:2341] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [7]:
keras.utils.set_random_seed(SEED)

review_inputs = keras.Input(shape=(MAXLEN,), dtype='int32', name='review_tokens')
padding_mask = layers.Lambda(lambda x: tf.not_equal(x, PAD_ID), name='padding_mask')(review_inputs)

embedding_layer = TokenAndPositionEmbedding(
    MAXLEN,
    NUM_WORDS,
    EMBED_DIM,
    name='token_and_position_embedding',
)
encoder_layer = TransformerEncoderBlock(
    EMBED_DIM,
    NUM_HEADS,
    FF_DIM,
    rate=0.1,
    name='transformer_encoder_block',
)

x = embedding_layer(review_inputs)
x = encoder_layer(x, mask=padding_mask)
pooled = layers.Lambda(lambda args: masked_average(args[0], args[1]), name='masked_average')(
    [x, padding_mask]
)
x = layers.Dropout(0.2)(pooled)
x = layers.Dense(32, activation='relu')(x)
x = layers.Dropout(0.2)(x)
review_outputs = layers.Dense(1, activation='sigmoid', name='sentiment')(x)

model = keras.Model(review_inputs, review_outputs, name='transformer_imdb_classifier')
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

trace_inputs = keras.Input(shape=(MAXLEN,), dtype='int32', name='trace_review_tokens')
trace_mask = layers.Lambda(lambda x: tf.not_equal(x, PAD_ID), name='trace_padding_mask')(trace_inputs)
trace_x = embedding_layer(trace_inputs)
trace_encoded, trace_scores = encoder_layer(trace_x, mask=trace_mask, return_attention_scores=True)
attention_trace_model = keras.Model(trace_inputs, [trace_encoded, trace_scores], name='imdb_trace_model')


/home/sorcerer/Projects/students-AI_math_essentials/.venv/lib/python3.12/site-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'masked_average' (of type Lambda) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


In [8]:
model.summary()


Model: "transformer_imdb_classifier"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ review_tokens       │ (None, 200)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ token_and_position… │ (None, 200, 32)   │    326,400 │ review_tokens[0]… │
│ (TokenAndPositionE… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ padding_mask        │ (None, 200)       │          0 │ review_tokens[0]… │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_encode… │ [(None, 200, 32), │      8,544 │ token_and_positi… │
│ (TransformerEncode… │ (None, 2, 200,    │            │ padding_mask[0][… │
│                     │ 200)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ masked_average      │ (None, 32)        │          0 │ transformer_enco… │
│ (Lambda)            │                   │            │ padding_mask[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_6 (Dropout) │ (None, 32)        │          0 │ masked_average[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 32)        │      1,056 │ dropout_6[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_7 (Dropout) │ (None, 32)        │          0 │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sentiment (Dense)   │ (None, 1)         │         33 │ dropout_7[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 336,033 (1.28 MB)

 Trainable params: 336,033 (1.28 MB)

 Non-trainable params: 0 (0.00 B)

## Как идёт обучение внутри эпохи

Для fast-mode этой лабораторной полезно смотреть не на абсолютный максимум метрики, а на три вещи:
- train loss убывает;
- validation accuracy растёт и не расходится с train;
- итоговая test accuracy заметно выше случайного уровня.


In [9]:
history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=0,
)

print(f"best val_accuracy: {max(history.history['val_accuracy']):.3f}")


best val_accuracy: 0.819


In [10]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.title('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='val')
plt.title('Accuracy')
plt.legend()
plt.tight_layout()
plt.show()


/tmp/ipykernel_280064/50891051.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Оценка и диагностика

Перед сдачей здесь должны одновременно выполняться все условия:
- `test_acc >= 0.75` на текущем subset;
- показан хотя бы один декодированный review;
- heatmap строится по первым содержательным токенам, а не по padded хвосту.

Сначала проверьте итоговую метрику, затем разберите один review и только после этого интерпретируйте attention trace.


In [11]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'test_loss = {test_loss:.4f}')
print(f'test_acc  = {test_acc:.4f}')


test_loss = 0.4143
test_acc  = 0.8300


In [12]:
word_index = keras.datasets.imdb.get_word_index()
reverse_word_index = {index + 3: word for word, index in word_index.items()}
reverse_word_index[0] = '[PAD]'
reverse_word_index[1] = '[START]'
reverse_word_index[2] = '[OOV]'
reverse_word_index[3] = '[UNUSED]'

def decode_review(token_ids):
    return [reverse_word_index.get(int(token), f'#{int(token)}') for token in token_ids]

sample_probs = model.predict(X_test[:32], verbose=0).ravel()
sample_index = int(np.argmax(np.abs(sample_probs - 0.5)))
sample_tokens = X_test[sample_index:sample_index + 1]
sample_target = y_test[sample_index]
sample_prob = float(model.predict(sample_tokens, verbose=0)[0, 0])

active_len = int(np.count_nonzero(sample_tokens[0]))
decoded_tokens = decode_review(sample_tokens[0][: min(active_len, 30)])

print('sample_index   :', sample_index)
print('true_label     :', int(sample_target))
print('predicted_prob :', round(sample_prob, 4))
print('decoded tokens :')
print(decoded_tokens)


sample_index   : 21
true_label     : 1
predicted_prob : 0.9964
decoded tokens :
['[START]', 'this', 'is', 'one', 'of', 'my', 'favourite', 'disney', 'films', 'it', 'has', 'everything', 'you', 'could', 'hope', 'for', 'in', 'a', 'disney', 'animation', 'cute', 'animals', 'great', 'songs', 'a', 'nasty', 'villain', 'and', 'lots', 'of']


In [13]:
_, attention_scores = attention_trace_model.predict(sample_tokens, verbose=0)
vis_len = min(active_len, 30)
mean_attention = attention_scores.mean(axis=1)[0][:vis_len, :vis_len]
visible_tokens = decode_review(sample_tokens[0][:vis_len])

token_importance = mean_attention.mean(axis=0)
top_positions = np.argsort(token_importance)[::-1][:8]
print('attention_scores raw shape:', attention_scores.shape)
print('mean_attention shape      :', mean_attention.shape)
print('most attended visible tokens:')
for pos in top_positions:
    print(pos, visible_tokens[pos], round(float(token_importance[pos]), 4))


attention_scores raw shape: (1, 2, 200, 200)
mean_attention shape      : (30, 30)
most attended visible tokens:
28 lots 0.0073
4 of 0.0073
29 of 0.0073
21 animals 0.0073
15 for 0.0073
25 nasty 0.0073
26 villain 0.0073
13 could 0.0073


In [14]:
plt.figure(figsize=(8, 6))
plt.imshow(mean_attention, cmap='magma', aspect='auto')
plt.colorbar(label='attention weight')
plt.xticks(range(vis_len), visible_tokens, rotation=90, fontsize=8)
plt.yticks(range(vis_len), visible_tokens, fontsize=8)
plt.xlabel('key positions')
plt.ylabel('query positions')
plt.title('Mean self-attention over heads on IMDB sample')
plt.tight_layout()
plt.show()


/tmp/ipykernel_280064/1103564216.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Если не получилось с первого раза

Сначала проверяйте не “Transformer ли плохой”, а:
- размер subset;
- длину `maxlen`;
- mask;
- classifier head;
- decode review для attention-визуализации.


## Если застряли: порядок диагностики

1. Проверить форму `X_train` и `y_train`.
2. Проверить `PAD = 0` и построение `padding_mask`.
3. Проверить shape после embedding и encoder block.
4. Проверить train/validation curves.
5. Проверить attention trace на одном review.


## Чек-лист перед сдачей

- `IMDB` загружен с ограничением словаря.
- После padding вход имеет форму `(N, T)`.
- Reuse-block из `ЛР01` используется как готовый блок, а не переписывается заново.
- Pooling учитывает mask.
- Есть итоговая оценка на test.
- `test_acc >= 0.75` на текущем subset.
- Есть хотя бы один декодированный review.
- Есть attention-визуализация по первым содержательным токенам review.


## Как использовать решение без самообмана

Неправильный путь:
- сразу открыть solution;
- скопировать reuse-block;
- получить working notebook без понимания, что именно вы перенесли из `ЛР01`.

Правильный путь:
- сначала собрать свою transfer-версию;
- потом свериться с walkthrough;
- потом пройти debugging playbook;
- сделать вторую попытку;
- и только после этого сравнить решение по блокам.


## Опционально после сдачи: мини-экзамен

1. Почему encoder-only Transformer уже подходит для `many-to-one` classification?
2. Зачем reuse toy-блока полезен методически?
3. Почему attention-карту review лучше показывать только на непустом фрагменте?
4. Как mask участвует и во внимании, и в pooling?


## Опционально после сдачи: что дальше

Этот notebook закрывает первую версию блока `03-Transformer`.

После него логично двигаться в одну из двух сторон:
- decoder-only Transformer и causal mask;
- полный encoder-decoder Transformer.

То есть теперь внимание уже не надстройка над RNN, а главный строительный блок модели.


## Опционально после сдачи: вопросы для самопроверки

## Типичные ошибки (симптом -> причина -> исправление)

- Обучение идёт слишком медленно -> слишком большой subset или `maxlen` -> вернуться к CPU-friendly конфигу.
- Accuracy слабая и нестабильная -> classifier head или mask собраны неверно -> проверить pooling и loss.
- Attention heatmap нечитаема -> показывается padded хвост или слишком длинный review -> обрезать до осмысленного фрагмента.
- Review декодируется странно -> не учтён offset словаря `IMDB` -> проверить `reverse_word_index`.
